In [ ]:
!pip install "transformers==4.41.2" "sentence-transformers==3.0.1" openai
!pip install -U datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 118.3 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [ ]:
data["train"][0]

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
 'label': 1}

In [ ]:
from transformers import pipeline

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

pipe = pipeline(model=model_path, tokenizer=model_path, return_all_scores=True, device="cuda:0")


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
output = pipe("This is a stupid movie.")
output

[{'label': 'negative', 'score': 0.9334307909011841}]

In [ ]:
output = pipe("I saw the movie")
output

[{'label': 'neutral', 'score': 0.8333644866943359}]

In [ ]:
output = pipe("I saw the movie")
output

In [ ]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

# Run inference
y_pred = []
for prediction_dict in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])): #pipe creates iterable of strings in 'text' for each data point
    # prediction_dict is expected to be a single dictionary like {'label': 'negative', 'score': 0.933}
    current_negative_score = 0.0
    current_positive_score = 0.0

    if prediction_dict['label'] == 'negative':
        current_negative_score = prediction_dict['score']
    elif prediction_dict['label'] == 'positive':
        current_positive_score = prediction_dict['score']
    # 'neutral' labels will result in both scores remaining 0.0, which means np.argmax will default to 0 (negative)

    assignment = np.argmax([current_negative_score, current_positive_score])
    y_pred.append(assignment)

100%|██████████| 1066/1066 [00:11<00:00, 95.22it/s] 


In [ ]:
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
  performance = classification_report(y_true, y_pred, target_names=["negative", "positive"])
  print(performance)

In [ ]:
evaluate_performance(data['test']['label'], y_pred)

              precision    recall  f1-score   support

    negative       0.68      0.94      0.79       533
    positive       0.91      0.56      0.69       533

    accuracy                           0.75      1066
   macro avg       0.79      0.75      0.74      1066
weighted avg       0.79      0.75      0.74      1066



In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

train_embeddings = model.encode(data["train"]["text"], show_progress_bar = True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar = True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/267 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

In [ ]:
train_embeddings.shape

(8530, 768)

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(random_state=0, max_iter=1000).fit(train_embeddings, data["train"]["label"])

y_pred = clf.predict(test_embeddings)
evaluate_performance(data['test']['label'], y_pred)

              precision    recall  f1-score   support

    negative       0.85      0.86      0.85       533
    positive       0.86      0.85      0.85       533

    accuracy                           0.85      1066
   macro avg       0.85      0.85      0.85      1066
weighted avg       0.85      0.85      0.85      1066



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

df = pd.DataFrame(np.hstack([train_embeddings, np.array(data["train"]["label"]).reshape(-1, 1)]))
averaged_target_embedding = df.groupby(768).mean().values

sim_matrix = cosine_similarity(test_embeddings, averaged_target_embedding)
y_pred = np.argmax(sim_matrix, axis=1)
evaluate_performance(data['test']['label'], y_pred)


              precision    recall  f1-score   support

    negative       0.85      0.84      0.84       533
    positive       0.84      0.85      0.84       533

    accuracy                           0.84      1066
   macro avg       0.84      0.84      0.84      1066
weighted avg       0.84      0.84      0.84      1066



In [ ]:
label_embeddings = model.encode(["A negative review", "A posiive review"])

sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)
evaluate_performance(data['test']['label'], y_pred)


              precision    recall  f1-score   support

    negative       0.59      0.89      0.71       533
    positive       0.78      0.39      0.52       533

    accuracy                           0.64      1066
   macro avg       0.68      0.64      0.62      1066
weighted avg       0.68      0.64      0.62      1066



In [ ]:
label_embeddings = model.encode(["A very negative review", "A very posiive review"])

sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)
evaluate_performance(data['test']['label'], y_pred)

              precision    recall  f1-score   support

    negative       0.78      0.78      0.78       533
    positive       0.78      0.78      0.78       533

    accuracy                           0.78      1066
   macro avg       0.78      0.78      0.78      1066
weighted avg       0.78      0.78      0.78      1066



In [ ]:
!pip install "transformers<5"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.20.1
    Uninstalling huggingface_hub-1.20.1:
      Successfully uninstalled huggingface_hub-1.20.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
from transformers import pipeline

model_path = "google/flan-t5-small"

pipe = pipeline("text2text-generation", model=model_path, tokenizer=model_path, device="cuda:0")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda:0


In [ ]:
prompt = "Is the following sentence positive or negative? "
data = data.map(lambda example: {"t5": prompt + example['text']})
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
})

In [ ]:
# Run inference
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "t5")), total=len(data["test"])):#or we can use pipe(data["test"]["t5"]) instead of KeyDataset
    text = output[0]["generated_text"]
    y_pred.append(0 if text == "negative" else 1)

100%|██████████| 1066/1066 [01:03<00:00, 16.82it/s]


In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

    negative       0.83      0.85      0.84       533
    positive       0.85      0.83      0.84       533

    accuracy                           0.84      1066
   macro avg       0.84      0.84      0.84      1066
weighted avg       0.84      0.84      0.84      1066



In [ ]:
from google import genai

client = genai.Client(api_key="x") # You need to replace "YOUR_API_KEY" with an actual API key

interaction = client.interactions.create(
    model="gemini-3.5-flash",
    input="Explain how AI works in a few words"
)
print(interaction.output_text)

**AI analyzes vast data to find patterns and make decisions.**


In [ ]:
import google.generativeai as genai

def gemini_generation(prompt_template, document, model_name="gemini-3.5-flash"):
    """Generate an output based on a prompt and an input document using google.generativeai."""
    # Configure the API key. This assumes the API key is available globally or passed.
    # Using the API key previously set in cell JPYfNzTQ-a_d.
    genai.configure(api_key="x")

    model = genai.GenerativeModel(model_name)

    # The prompt and document are combined into a single user input.
    user_content = prompt_template.replace("[DOCUMENT]", document)

    # For generate_content, a list of parts with 'role' is the standard way to send messages.
    messages = [
        {"role": "user", "parts": [user_content]}
    ]

    # Making the actual API call
    try:
        response = model.generate_content(
            messages,
            generation_config=genai.types.GenerationConfig(temperature=0)
        )
        # Extract the text from the response
        if response.candidates and response.candidates[0].content.parts:
            return response.candidates[0].content.parts[0].text
        else:
            return "No text generated in response."
    except Exception as e:
        return f"Error during generation: {e}"


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# Define a prompt template as a base
prompt = """Predict whether the following document is a positive or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0. Do not give any other answers.
"""

# Predict the target using Gemini
document = "unpretentious , charming , quirky , original"
gemini_generation(prompt, document)

'1'

In [ ]:
predictions = [gemini_generation(prompt, doc) for doc in tqdm(data["test"]["text"])]

100%|██████████| 1066/1066 [07:41<00:00,  2.31it/s]


In [ ]:
# Initialize lists for valid predictions and true labels
valid_y_pred = []
valid_y_true = []
failed_predictions_count = 0

# Iterate through predictions and data to filter valid ones
for i, pred_str in enumerate(predictions):
    if pred_str == '0':
        valid_y_pred.append(0)
        valid_y_true.append(data["test"]["label"][i])
    elif pred_str == '1':
        valid_y_pred.append(1)
        valid_y_true.append(data["test"]["label"][i])
    else:
        # Increment counter for failed predictions
        failed_predictions_count += 1

# Report on failed predictions
print(f"Number of failed or invalid predictions: {failed_predictions_count} out of {len(predictions)}")

# Evaluate performance only on valid predictions
if valid_y_pred:
    evaluate_performance(valid_y_true, valid_y_pred)
else:
    print("No valid predictions to evaluate.")

Number of failed or invalid predictions: 1050 out of 1066
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         0
    positive       1.00      0.94      0.97        16

    accuracy                           0.94        16
   macro avg       0.50      0.47      0.48        16
weighted avg       1.00      0.94      0.97        16



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
